In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Tours

In [ ]:
def total_tours(data1, data2, tag='PSRC Region'):
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    if int(model_year) > 2023:
        if tag == 'PSRC Region':
            Tour_2_total *= TOUR_FACTOR_PSRC_2023
        elif tag == 'BKR':
            Tour_2_total *= TOUR_FACTOR_BKR_2023

    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = Tour_1_total
    tpp[f'{survey_year}Survey'] = Tour_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.0f}',
        f'{survey_year}Survey': '{:,.0f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.0f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [ ]:
total_tours(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_tours(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour per Person

In [ ]:
def tour_per_person(data1, data2, data3=data_fullsurvey, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_3_total = get_total(data3['Person']['psexpfac'])
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    if int(model_year) > 2023:
        if tag == 'PSRC Region':
            Tour_2_total *= TOUR_FACTOR_PSRC_2023
        elif tag == 'BKR':
            Tour_2_total *= TOUR_FACTOR_BKR_2023

    ##Tours per person
    tpp1 = Tour_1_total / Person_1_total
    tpp2 = Tour_2_total / Person_3_total
    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.2f}',
        f'{survey_year}Survey': '{:,.2f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.2f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [ ]:
tour_per_person(data1=data_daysim, data2=data_survey, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
tour_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Tours per Person by Person Type

In [ ]:
from collections import OrderedDict

def tour_by_pptyp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose and Person Type/Number of Stops
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    PersonsDay1 = pd.merge(data1['Person'][['hhno', 'pno', 'pptyp', 'psexpfac']], data1['PersonDay'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    PersonsDay2 = pd.merge(data2['Person'][['hhno', 'pno', 'pptyp', 'psexpfac']], data2['PersonDay_cloned'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    # calculate the percentage of each number of stops by purpose
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    intermediate_stopsbypurp1 = data1['Tour'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    intermediate_stopsbypurp2 = data2['Tour_cloned'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    # retain only the first 5 stops
    intermediate_stopsbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['all_stops']<=5]
    intermediate_stopsbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['all_stops']<=5]

    for purpose in data1['Tour']['pdpurp'].value_counts().index:
        if purpose == 'Work':
            tc = 'wktours'
            sc = 'wkstops'
        elif purpose == 'Social':
            tc = 'sotours'
            sc = 'sostops'
        elif purpose == 'School':
            tc = 'sctours'
            sc = 'scstops'
        elif purpose == 'Escort':
            tc = 'estours'
            sc = 'esstops'
        elif purpose == 'Personal Business':
            tc = 'pbtours'
            sc = 'pbstops'
        elif purpose == 'Shop':
            tc = 'shtours'
            sc = 'shstops'
        elif purpose == 'Meal':
            tc = 'mltours'
            sc = 'mlstops'
        #Merge a column to PersonsDay for the current purpose
        PersonsDay1 = PersonsDay1.merge(data1['PersonDay'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        PersonsDay2 = PersonsDay2.merge(data2['PersonDay_cloned'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        toursPersPurp1 = weighted_average(PersonsDay1, tc, 'psexpfac', 'pptyp')
        toursPersPurp2 = weighted_average(PersonsDay2, tc, 'psexpfac', 'pptyp')
        #Delete added column to make future iterations faster
        PersonsDay1.drop(columns = [tc], inplace = True)
        PersonsDay2.drop(columns = [tc], inplace = True)
        items = OrderedDict(((name1, toursPersPurp1), (name2, toursPersPurp2)))
        toursPersPurp = pd.DataFrame.from_dict(items)
        toursPersPurp = get_differences(toursPersPurp, name1, name2, 3)
        toursPersPurp = recode_index(toursPersPurp, 'pptyp','Person Type')
        toursPersPurp = toursPersPurp.loc[ptype_cat.values(), :]
        # display table
        def pct_fmt(x):
            return 'nan' if pd.isna(x) else f'{x:,.2f}%'
        def pct_compare_fmt(x):
            return 'nan' if pd.isna(x) else f'{x:,.1f}%'
        display(toursPersPurp.style.format({
            name1: '{:,.3f}',
            name2: '{:,.3f}',
            f'Difference ({name1} - {name2})': '{:,.2f}',
            f'% Difference ({name1} - {name2})': pct_compare_fmt,
        }))
        # bar plot
        fig = px.bar(
            toursPersPurp.reset_index(),
            x='Person Type',
            y=[name1, name2],
            barmode='group',
            title=f'{purpose} Tours by Person Type'
        )
        fig.update_layout(yaxis_title=f'{purpose} Tours per Person', xaxis_title='Person Type', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig.show()

        #Number of stops by purpose
        ps = pd.DataFrame()
        # fill missing 'all_stops' values with 0 for both intermediate_stopsbypurp1 and intermediate_stopsbypurp2
        imstpbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['pdpurp']==purpose].copy(deep=True)
        imstpbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['pdpurp']==purpose].copy(deep=True)
        # calculate percentage
        imstpbypurp1['percentage'] = imstpbypurp1['toexpfac'] / imstpbypurp1['toexpfac'].sum() * 100
        imstpbypurp2['percentage'] = imstpbypurp2['toexpfac'] / imstpbypurp2['toexpfac'].sum() * 100
        imstpbypurp1 = imstpbypurp1[imstpbypurp1['all_stops']<=5]
        imstpbypurp2 = imstpbypurp2[imstpbypurp2['all_stops']<=5]
        imstpbypurp1 = imstpbypurp1.set_index('all_stops').reindex(range(0, 6), fill_value=0).reset_index()
        imstpbypurp2 = imstpbypurp2.set_index('all_stops').reindex(range(0, 6), fill_value=0).reset_index()
        ps['% of Tours (' + name1 + ')'] = list(imstpbypurp1['percentage'])
        ps['% of Tours (' + name2 + ')'] = list(imstpbypurp2['percentage'])
        ps[purpose + ' Tours'] = range(0, 6)
        ps = ps.set_index(purpose + ' Tours')
        ps = get_differences( ps, '% of Tours (' + name1 + ')', '% of Tours (' + name2 + ')', 2)
        # table
        display(ps.style.format({
            f'% of Tours ({name1})': pct_compare_fmt,
            f'% of Tours ({name2})': pct_compare_fmt,
            f'Difference (% of Tours ({name1}) - % of Tours ({name2}))': pct_compare_fmt,
            f'% Difference (% of Tours ({name1}) - % of Tours ({name2}))': pct_compare_fmt
        }))

        # figure
        fig_stops = px.bar(
            ps.reset_index(),
            x=ps.index.name,
            y=[f'% of Tours ({name1})', f'% of Tours ({name2})'],
            barmode='group',
            title=f'Number of Stops per Tour by Purpose ({purpose}, {tag})'
        )
        fig_stops.update_layout(yaxis_title='Percentage of Tours', xaxis_title='Number of Stops', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig_stops.update_yaxes(ticksuffix='%')
        fig_stops.show()

In [ ]:
tour_by_pptyp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
tour_by_pptyp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tours per Person by Purpose

In [ ]:
def tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['DaysimOutputs'] = tpbp1
    tpbp[f'{survey_year}Survey'] = tpbp2
    tpbp = get_differences(tpbp, 'DaysimOutputs', '{survey_year}Survey', 2)
    tpbp = recode_index(tpbp, 'pdpurp', 'Tour Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    def pct_fmt(x):
        return 'nan' if pd.isna(x) else f'{x:,.1f}%'
    display(tpbp.style.format({
        'DaysimOutputs': '{:,.2f}',
        f'{survey_year}Survey': '{:,.2f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.2f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": pct_fmt,
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Purpose',
        y=['DaysimOutputs', f'{survey_year}Survey'],
        barmode='group',
        title=f'Tours per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tours per Person by Mode

In [ ]:
def tours_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Mode
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['DaysimOutputs'] = tpbp1
    tpbp[f'{survey_year}Survey'] = tpbp2
    tpbp = get_differences(tpbp, 'DaysimOutputs', '{survey_year}Survey', 3)
    tpbp = recode_index(tpbp, 'tmodetp', 'Tour Mode')
    tpbp = tpbp.loc[mode_cat.values()]
    # table
    def pct_fmt(x):
        return 'nan' if pd.isna(x) else f'{x:,.1f}%'
    display(tpbp.style.format({
        'DaysimOutputs': '{:,.3f}',
        f'{survey_year}Survey': '{:,.3f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.3f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": pct_fmt,
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Mode',
        y=['DaysimOutputs', f'{survey_year}Survey'],
        barmode='group',
        title=f'Tours per Person by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_per_ps_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

In [ ]:
from collections import OrderedDict

def tour_by_mode_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose and Person Type/Number of Stops
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    PersonsDay1 = pd.merge(data1['Tour'][['hhno', 'pno', 'tmodetp']], data1['PersonDay'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    PersonsDay2 = pd.merge(data2['Tour_cloned'][['hhno', 'pno', 'tmodetp']], data2['PersonDay_cloned'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    # calculate the percentage of each number of stops by purpose
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    intermediate_stopsbypurp1 = data1['Tour'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    intermediate_stopsbypurp2 = data2['Tour_cloned'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    # retain only the first 5 stops
    intermediate_stopsbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['all_stops']<=5]
    intermediate_stopsbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['all_stops']<=5]

    for purpose in data1['Tour']['pdpurp'].value_counts().index:
        if purpose == 'Work':
            tc = 'wktours'
            sc = 'wkstops'
        elif purpose == 'Social':
            tc = 'sotours'
            sc = 'sostops'
        elif purpose == 'School':
            tc = 'sctours'
            sc = 'scstops'
        elif purpose == 'Escort':
            tc = 'estours'
            sc = 'esstops'
        elif purpose == 'Personal Business':
            tc = 'pbtours'
            sc = 'pbstops'
        elif purpose == 'Shop':
            tc = 'shtours'
            sc = 'shstops'
        elif purpose == 'Meal':
            tc = 'mltours'
            sc = 'mlstops'
        #Merge a column to PersonsDay for the current purpose
        PersonsDay1 = PersonsDay1.merge(data1['PersonDay'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        PersonsDay2 = PersonsDay2.merge(data2['PersonDay_cloned'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        toursPersPurp1 = weighted_average(PersonsDay1, tc, 'pdexpfac', 'tmodetp')
        toursPersPurp2 = weighted_average(PersonsDay2, tc, 'pdexpfac', 'tmodetp')
        #Delete added column to make future iterations faster
        PersonsDay1.drop(columns = [tc], inplace = True)
        PersonsDay2.drop(columns = [tc], inplace = True)
        items = OrderedDict(((name1, toursPersPurp1), (name2, toursPersPurp2)))
        toursPersPurp = pd.DataFrame.from_dict(items)
        toursPersPurp = get_differences(toursPersPurp, name1, name2, 2)
        toursPersPurp = recode_index(toursPersPurp, 'tmodetp','Mode')
        # Retain only rows where the index (mode) is not 'Other'
        toursPersPurp = toursPersPurp[toursPersPurp.index != 'Other']
        toursPersPurp = toursPersPurp.loc[mode_cat.values(), :]
        # display table
        def pct_fmt(x):
            return 'nan' if pd.isna(x) else f'{x:,.1f}%'
        display(toursPersPurp.style.set_caption(f'Purpose: {purpose}').format({
            name1: '{:,.2f}',
            name2: '{:,.2f}',
            f'Difference ({name1} - {name2})': '{:,.2f}',
            f'% Difference ({name1} - {name2})': pct_fmt,
        }))
        # bar plot
        fig = px.bar(
            toursPersPurp.reset_index(),
            x='Mode',
            y=[name1, name2],
            barmode='group',
            title=f'{purpose} Tours by Mode'
        )
        fig.update_layout(yaxis_title=f'{purpose} Tours per Person', xaxis_title='Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig.show()

In [ ]:
# :::{.panel-tabset}

# ### PSRC Region

In [ ]:
# tour_by_mode_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
### BKR

In [ ]:
# tour_by_mode_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

In [ ]:
# :::

## Tour Share by Purpose

In [ ]:
def pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Purpose
    tour_1_total = get_total(data1['Tour']['toexpfac'])
    tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'pdpurp', 'Tour Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Purpose',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Tours by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Tours', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [ ]:
pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
pc_tour_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Share by Mode

In [ ]:
def pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Mode
    tour_1_total = get_total(data1['Tour']['toexpfac'])
    tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'tmodetp', 'Tour Mode')
    ptbp = ptbp.loc[mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Mode',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Tour Share by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tour Share', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [ ]:
pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
pc_tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Distance by Purpose

In [ ]:
def tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.2f}',
        f'Average Tour Length ({name2})': '{:,.2f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Distance by Mode

In [ ]:
def tour_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    #Average Distance by Tour Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.2f}',
        f'Average Tour Length ({name2})': '{:,.2f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Travel Time by Purpose

In [ ]:
def tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Travel Time by Mode

In [ ]:
def tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[mode_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tours by District

In [ ]:
##Tours per Person by Purpose and Person Type/Number of Stops
data1=data_daysim
data2=data_survey
name1 = 'DaysimOutputs'
name2 = f'{survey_year}Survey' 
# calculate the percentage of each number of stops by purpose
data1['Household'] = data1['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data1['Tour'] = data1['Tour'].merge(data1['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
data2['Household'] = data2['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data2['Tour_cloned'] = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
todist1 = data1['Tour'].groupby(by='DistrictFlowName')['toexpfac'].sum()
todist2 = data2['Tour_cloned'].groupby(by='DistrictFlowName')['toexpfac'].sum()
df_compare = pd.concat([todist1, todist2], axis=1)
df_compare.columns = [name1, name2]
df_compare['Difference'] = df_compare[name1] - df_compare[name2]
df_compare['% Difference'] = (df_compare['Difference'] / df_compare[name2]) * 100
display(df_compare.loc[district_flow_name.values()].style.format({
    name1: '{:,.0f}',
    name2: '{:,.0f}',
    'Difference': '{:,.0f}',
    '% Difference': '{:,.1f}%'
}))

In [ ]:
def tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Tour Purpose',
                        gp1_list=[], gp2_list=[]):
    tour_by_district_purpose1 = data1['Tour'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    tour_by_district_purpose2 = data2['Tour_cloned'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    tour_by_district_purpose1 = tour_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose2 = tour_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose1.columns.name = gp2_label
    tour_by_district_purpose2.columns.name = gp2_label
    tour_by_district_purpose1.index.name = gp1_label
    tour_by_district_purpose2.index.name = gp1_label
    display(tour_by_district_purpose1.style.format('{:,.0f}').set_caption("DaysimOutputs"))
    display(tour_by_district_purpose2.style.format('{:,.0f}').set_caption(f"{survey_year}Survey"))
    percent_diff = (tour_by_district_purpose1 - tour_by_district_purpose2) / tour_by_district_purpose2 * 100
    display(percent_diff.style.format('{:,.1f}%').set_caption("Percentage Difference (DaysimOutputs - Survey)"))

In [ ]:
def tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Tour Purpose',
                        gp1_list=[], gp2_list=[]):
    tour_by_district_purpose1 = data1['Tour'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    tour_by_district_purpose2 = data2['Tour_cloned'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    tour_by_district_purpose1 = tour_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose2 = tour_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose1.columns.name = gp2_label
    tour_by_district_purpose2.columns.name = gp2_label
    tour_by_district_purpose1.index.name = gp1_label
    tour_by_district_purpose2.index.name = gp1_label
    tour_by_district_purpose1 = tour_by_district_purpose1.div(tour_by_district_purpose1.sum(axis=0), axis=1) * 100
    tour_by_district_purpose2 = tour_by_district_purpose2.div(tour_by_district_purpose2.sum(axis=0), axis=1) * 100
    def pct_fmt(x):
        return 'nan' if pd.isna(x) else f'{x:,.1f}%'
    display(tour_by_district_purpose1.style.format(pct_fmt).set_caption("DaysimOutputs"))
    display(tour_by_district_purpose2.style.format(pct_fmt).set_caption(f"{survey_year}Survey"))

## Tours by District by Purpose

In [ ]:
# :::{.panel-tabset}

# ### Number of Tours

In [ ]:
# tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
#                         gp1='pdpurp', gp2='DistrictFlowName', 
#                         gp1_label='Tour Purpose', gp2_label='DistrictFlowName',
#                         gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

In [ ]:
### Tour Share

In [ ]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='pdpurp', gp2='DistrictFlowName', 
                        gp1_label='Tour Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

In [ ]:
# :::

## Tours by District by Mode

In [ ]:
# :::{.panel-tabset}

# ### Number of Tours

In [ ]:
# tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
#                     gp1='DistrictFlowName', gp2='tmodetp', 
#                     gp1_label='DistrictFlowName', gp2_label='Tour Mode',
#                     gp1_list=district_flow_name.values(), gp2_list=mode_cat.values())

In [ ]:
### Tour Share

In [ ]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='tmodetp', 
                    gp1_label='DistrictFlowName', gp2_label='Tour Mode',
                    gp1_list=district_flow_name.values(), gp2_list=mode_cat.values())

In [ ]:
# :::

## Tours by Purpose by Auto Sufficiency

In [ ]:
# :::{.panel-tabset}

# ### Number of Tours

In [ ]:
def classify_auto_sufficiency(row):
    if row['hhvehs'] == 0:
        return '0 veh'
    elif row['hhvehs'] < row['driver']:
        return 'autos < drivers'
    elif row['hhvehs'] == row['driver']:
        return 'autos = drivers'
    else:
        return 'autos > drivers'

def get_auto_sufficiency(data1=data_daysim, data2=data_survey):
    data1['Trip']['driver'] = (data1['Trip']['dorp'] == 'Driver').astype(int)
    data2['Trip_cloned']['driver'] = (data2['Trip_cloned']['dorp'] == 'Driver').astype(int)

    _driver1 = data1['Trip'][['hhno', 'pno', 'driver']].groupby(by=['hhno', 'pno'])['driver'].sum().reset_index()
    _driver1['driver'] = (_driver1['driver'] > 0).astype(int)
    driver1 = _driver1.groupby(by='hhno')['driver'].sum()

    _driver2 = data2['Trip_cloned'][['hhno', 'pno', 'driver']].groupby(by=['hhno', 'pno'])['driver'].sum().reset_index()
    _driver2['driver'] = (_driver2['driver'] > 0).astype(int)
    driver2 = _driver2.groupby(by='hhno')['driver'].sum()

    data1['Tour'] = data1['Tour'].merge(driver1, on='hhno', how='left')
    data2['Tour_cloned'] = data2['Tour_cloned'].merge(driver2, on='hhno', how='left')

    data1['Tour']['auto_sufficiency'] = data1['Tour'][['hhvehs', 'driver']].apply(classify_auto_sufficiency, axis=1)
    data2['Tour_cloned']['auto_sufficiency'] = data2['Tour_cloned'][['hhvehs', 'driver']].apply(classify_auto_sufficiency, axis=1)
    return data1, data2

In [ ]:
autosuf_labels = ['0 veh', 'autos < drivers', 'autos = drivers', 'autos > drivers']
data_daysim, data_survey = get_auto_sufficiency(data1=data_daysim, data2=data_survey)

In [ ]:
# tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
#                     gp1='pdpurp', gp2='auto_sufficiency', 
#                     gp1_label='Tour Purpose', gp2_label='Auto Sufficiency',
#                     gp1_list=pdpurp_cat.values(), gp2_list=autosuf_labels)

In [ ]:
### Tour Share

In [ ]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='auto_sufficiency', 
                    gp1_label='Tour Purpose', gp2_label='Auto Sufficiency',
                    gp1_list=pdpurp_cat.values(), gp2_list=autosuf_labels)

In [ ]:
# :::

## Tours by Purpose by Income Level

In [ ]:
# :::{.panel-tabset}

# ### Number of Tours

In [ ]:
def get_income_group(data1=data_daysim, data2=data_survey):
    data1['Tour']['income_group'] = pd.cut(data1['Tour']['hhincome'], bins=income_bins, labels=income_labels, right=False)
    data2['Tour_cloned']['income_group'] = pd.cut(data2['Tour_cloned']['hhincome'], bins=income_bins, labels=income_labels, right=False)
    return data1, data2

In [ ]:
income_bins = [0, 25000, 45000, 75000, float('inf')]
income_labels = ['Less than 25,000', '$25,000-$44,999', '$45,000-74,999', 'More than $75,000']
data_daysim, data_survey = get_income_group(data1=data_daysim, data2=data_survey)

In [ ]:
# tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
#                     gp1='pdpurp', gp2='income_group', 
#                     gp1_label='Tour Purpose', gp2_label='Income Level',
#                     gp1_list=pdpurp_cat.values(), gp2_list=income_labels)

In [ ]:
### Tour Share

In [ ]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='income_group', 
                    gp1_label='Tour Purpose', gp2_label='Income Level',
                    gp1_list=pdpurp_cat.values(), gp2_list=income_labels)

In [ ]:
# :::